# Assignment No. 4 — Denoising Diffusion Probabilistic Model (DDPM)
**Course:** Generative AI (AI4009) | **Semester:** Spring 2026

**Overview:** This notebook implements a full DDPM pipeline from scratch using pure PyTorch.
- Part 1: Config, Data & Forward Process
- Part 2: U-Net Architecture
- Part 3: Training Loop
- Part 4: Sampling, Generation & Metrics
- Part 5: Gradio App (app.py for Hugging Face Spaces)

---
## Cell 0 — Install Dependencies

In [1]:
# Run once on Kaggle to ensure all packages are available
!pip install -q scikit-image gradio

---
## PART 1 — Configuration, Data & Forward Diffusion Process

In [2]:
# ─────────────────────────────────────────────────────────────
#  Cell 1.1 — Imports
# ─────────────────────────────────────────────────────────────
import os, math, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from torchvision.utils import make_grid
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim
import glob

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPU count       : {torch.cuda.device_count()}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU count       : 2


In [5]:
# ─────────────────────────────────────────────────────────────
#  Cell 1.2 — Global Configuration
# ─────────────────────────────────────────────────────────────
class Config:
    # ── Data ──────────────────────────────────────────────────
    # Update DATA_DIR to the path of your Kaggle dataset images.
    # Example for CelebA-HQ: "/kaggle/input/celebahq256-images-only/data256x256"
    DATA_DIR    = "/kaggle/input/datasets/denislukovnikov/celebahq256-images-only/celebahq256_imgs"
    IMG_SIZE    = 128          # 128x128 minimum; set to 256 for higher quality
    IMG_EXTS    = ("*.jpg", "*.jpeg", "*.png")

    # ── Training ──────────────────────────────────────────────
    BATCH_SIZE  = 8         # Keep small to avoid T4 OOM
    EPOCHS      = 5          # Increase for better quality
    LR          = 2e-4
    GRAD_CLIP   = 1.0          # Max gradient norm; set None to disable
    NUM_WORKERS = 0

    # ── Diffusion ─────────────────────────────────────────────
    T           = 300          # Total diffusion timesteps (200-500 recommended)
    BETA_START  = 1e-4         # Starting noise level
    BETA_END    = 0.02         # Ending noise level

    # ── U-Net ─────────────────────────────────────────────────
    BASE_CHANNELS = 64         # Channel progression: 64 → 128 → 256

    # ── Misc ──────────────────────────────────────────────────
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
    SAVE_PATH   = "ddpm_model.pth"
    SEED        = 42

cfg = Config()
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print(f"Device: {cfg.DEVICE}  |  Image size: {cfg.IMG_SIZE}x{cfg.IMG_SIZE}  |  T={cfg.T}")

Device: cuda  |  Image size: 128x128  |  T=300


In [6]:
# ─────────────────────────────────────────────────────────────
#  Cell 1.3 — Custom Dataset & DataLoader
# ─────────────────────────────────────────────────────────────
class ImageFolderDataset(Dataset):
    """Loads every image found (recursively) under root_dir."""

    def __init__(self, root_dir: str, img_size: int):
        self.paths = []
        for ext in Config.IMG_EXTS:
            self.paths += glob.glob(os.path.join(root_dir, "**", ext), recursive=True)
        if len(self.paths) == 0:
            raise FileNotFoundError(f"No images found in {root_dir}")
        print(f"Found {len(self.paths)} images in {root_dir}")

        # Transforms: resize → centre-crop → tensor → normalise to [-1, 1]
        self.transform = transforms.Compose([
            transforms.Resize(img_size + 16),          # slight oversize for crop
            transforms.CenterCrop(img_size),
            transforms.ToTensor(),                      # [0, 1]
            transforms.Normalize([0.5, 0.5, 0.5],      # → [-1, 1]
                                 [0.5, 0.5, 0.5]),
        ])

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)


# Build dataset & loader
dataset = ImageFolderDataset(cfg.DATA_DIR, cfg.IMG_SIZE)
loader  = DataLoader(
    dataset,
    batch_size  = cfg.BATCH_SIZE,
    shuffle     = True,
    num_workers = cfg.NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True,
)
print(f"Loader: {len(loader)} batches per epoch.")

KeyboardInterrupt: 

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 1.4 — Noise Schedule & Forward Diffusion
# ─────────────────────────────────────────────────────────────
class LinearNoiseSchedule:
    """
    Pre-computes all quantities needed for q(x_t | x_0):

        beta_t         — linear schedule from BETA_START to BETA_END
        alpha_t        — 1 - beta_t
        alpha_hat_t    — cumulative product of alpha  (ᾱ_t)
        sqrt_alpha_hat — √ᾱ_t   (multiplied with x_0)
        sqrt_one_minus_alpha_hat — √(1-ᾱ_t)  (multiplied with ε)
    """

    def __init__(self, T: int, beta_start: float, beta_end: float, device: str):
        self.T      = T
        self.device = device

        # β schedule: linearly spaced from β_start → β_end
        self.betas  = torch.linspace(beta_start, beta_end, T).to(device)

        # α_t = 1 - β_t
        self.alphas = 1.0 - self.betas

        # ᾱ_t = ∏ α_s  for s=1..t
        self.alpha_hat = torch.cumprod(self.alphas, dim=0)

        # Frequently needed square roots
        self.sqrt_alpha_hat           = torch.sqrt(self.alpha_hat)
        self.sqrt_one_minus_alpha_hat = torch.sqrt(1.0 - self.alpha_hat)

    # ── forward diffusion ─────────────────────────────────────
    def q_sample(self, x0: torch.Tensor, t: torch.Tensor):
        """
        Sample a noisy image x_t from x_0 at timestep t using the
        closed-form forward formula:
            x_t = √ᾱ_t · x_0  +  √(1-ᾱ_t) · ε,   ε ~ N(0,I)

        Args:
            x0  : (B, C, H, W)  clean images in [-1, 1]
            t   : (B,)          integer timesteps in [0, T-1]
        Returns:
            x_t : noisy images  (same shape as x0)
            eps : the noise that was added
        """
        eps   = torch.randn_like(x0)                         # ε ~ N(0,I)

        # Index schedule tensors for each sample in the batch
        sqrt_ah     = self.sqrt_alpha_hat[t][:, None, None, None]
        sqrt_1m_ah  = self.sqrt_one_minus_alpha_hat[t][:, None, None, None]

        x_t = sqrt_ah * x0 + sqrt_1m_ah * eps               # reparameterisation trick
        return x_t, eps

    def sample_timesteps(self, batch_size: int) -> torch.Tensor:
        """Uniform random timesteps in [1, T-1] for a batch."""
        return torch.randint(1, self.T, (batch_size,), device=self.device)


noise_schedule = LinearNoiseSchedule(cfg.T, cfg.BETA_START, cfg.BETA_END, cfg.DEVICE)
print("Noise schedule ready.")
print(f"  β range : [{noise_schedule.betas[0]:.5f}, {noise_schedule.betas[-1]:.4f}]")
print(f"  ᾱ range : [{noise_schedule.alpha_hat[0]:.4f}, {noise_schedule.alpha_hat[-1]:.6f}]")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 1.5 — Visualise Forward Diffusion (≥ 5 steps)
# ─────────────────────────────────────────────────────────────
def tensor_to_numpy_img(t: torch.Tensor) -> np.ndarray:
    """Convert a [-1,1] CHW tensor to a uint8 HWC numpy array."""
    t = t.detach().cpu().clamp(-1, 1)
    t = (t + 1) / 2              # → [0, 1]
    return (t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def visualise_forward_diffusion(x0: torch.Tensor, schedule: LinearNoiseSchedule,
                                n_steps: int = 7):
    """
    Display the original image alongside noisy versions at evenly-spaced
    timesteps from 1 → T-1.
    """
    x0_gpu = x0.unsqueeze(0).to(schedule.device)            # (1, C, H, W)
    steps  = np.linspace(1, schedule.T - 1, n_steps, dtype=int).tolist()

    fig, axes = plt.subplots(1, n_steps + 1, figsize=(3 * (n_steps + 1), 3.5))
    fig.suptitle("Forward Diffusion Process  (x₀  →  xₜ)", fontsize=14, y=1.02)

    axes[0].imshow(tensor_to_numpy_img(x0))
    axes[0].set_title("Original\n(t=0)", fontsize=9)
    axes[0].axis("off")

    for i, t_val in enumerate(steps):
        t_tensor = torch.tensor([t_val], device=schedule.device)
        x_t, _   = schedule.q_sample(x0_gpu, t_tensor)
        axes[i + 1].imshow(tensor_to_numpy_img(x_t.squeeze(0)))
        axes[i + 1].set_title(f"t = {t_val}", fontsize=9)
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.savefig("forward_diffusion.png", bbox_inches="tight")
    plt.show()


# Grab one sample and visualise
sample_batch = next(iter(loader))
visualise_forward_diffusion(sample_batch[0], noise_schedule, n_steps=7)

---
## PART 2 — U-Net Architecture (Reverse Process)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 2.1 — Sinusoidal Time-Step Embedding
# ─────────────────────────────────────────────────────────────
class SinusoidalTimeEmbedding(nn.Module):
    """
    Encodes integer timestep t into a fixed-dimensional vector using
    sine / cosine positional encoding, then projects it through an MLP.

    This embedding is injected into every residual block so the network
    knows 'how noisy' its current input is.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # Two-layer MLP after the sinusoidal encoding
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t : (B,) integer timesteps
        Returns:
            emb : (B, dim) time embeddings
        """
        half_dim = self.dim // 2
        # Exponentially spaced frequencies
        freq = torch.exp(
            -math.log(10_000) * torch.arange(half_dim, device=t.device) / (half_dim - 1)
        )
        args = t[:, None].float() * freq[None, :]    # (B, half_dim)
        emb  = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
        return self.mlp(emb)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 2.2 — Residual Block with Time Injection
# ─────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    """
    A basic residual block that:
    1. Applies GroupNorm → SiLU → Conv  (first path)
    2. Adds a learned scale/shift from the time embedding
    3. Applies GroupNorm → SiLU → Dropout → Conv  (second path)
    4. Adds a residual shortcut (1×1 conv if channel dims differ)
    """

    def __init__(self, in_ch: int, out_ch: int, time_emb_dim: int,
                 groups: int = 8, dropout: float = 0.1):
        super().__init__()

        # Time embedding → scale & shift for the second norm
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch * 2),  # 2× for scale AND shift
        )

        # First conv block
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)

        # Second conv block
        self.norm2   = nn.GroupNorm(groups, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2   = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        # Shortcut: 1×1 conv to match channel dims (identity if equal)
        self.shortcut = (
            nn.Conv2d(in_ch, out_ch, 1)
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x     : (B, in_ch, H, W)   feature map
            t_emb : (B, time_emb_dim)  time embedding
        """
        h = self.conv1(F.silu(self.norm1(x)))

        # Inject time: compute scale γ and shift β, apply as feature-wise affine
        scale, shift = self.time_proj(t_emb).chunk(2, dim=-1)   # each (B, out_ch)
        h = h * (scale[:, :, None, None] + 1) + shift[:, :, None, None]

        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.shortcut(x)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 2.3 — Downsample & Upsample Helpers
# ─────────────────────────────────────────────────────────────
class Downsample(nn.Module):
    """Halves spatial resolution using a strided convolution."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    """Doubles spatial resolution via nearest-neighbour + conv."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 2.4 — Full U-Net
#             Channel progression: 64 → 128 → 256
# ─────────────────────────────────────────────────────────────
class UNet(nn.Module):
    """
    Simplified U-Net for DDPM noise prediction.

    Architecture:
        Input  (B, 3, H, W)  + timestep (B,)
          ↓
        Encoder: 64 → 128 → 256  (downsampling blocks)
          ↓
        Bottleneck: 256 → 256
          ↓
        Decoder: 256 → 128 → 64  (upsampling blocks + skip connections)
          ↓
        Output (B, 3, H, W)  — predicted noise ε
    """

    def __init__(self, in_channels: int = 3, base_channels: int = 64,
                 time_emb_dim: int = 256):
        super().__init__()

        ch  = base_channels                  # 64
        ch2 = base_channels * 2              # 128
        ch4 = base_channels * 4              # 256

        # ── Time Embedding ────────────────────────────────────
        self.time_emb = SinusoidalTimeEmbedding(time_emb_dim)

        # ── Initial Projection ────────────────────────────────
        self.init_conv = nn.Conv2d(in_channels, ch, 3, padding=1)

        # ── Encoder ───────────────────────────────────────────
        # Level 1 : ch  (H × W)
        self.enc1_res1 = ResidualBlock(ch,  ch,  time_emb_dim)
        self.enc1_res2 = ResidualBlock(ch,  ch,  time_emb_dim)
        self.down1     = Downsample(ch)

        # Level 2 : ch2  (H/2 × W/2)
        self.enc2_res1 = ResidualBlock(ch,  ch2, time_emb_dim)
        self.enc2_res2 = ResidualBlock(ch2, ch2, time_emb_dim)
        self.down2     = Downsample(ch2)

        # Level 3 : ch4  (H/4 × W/4)
        self.enc3_res1 = ResidualBlock(ch2, ch4, time_emb_dim)
        self.enc3_res2 = ResidualBlock(ch4, ch4, time_emb_dim)
        self.down3     = Downsample(ch4)

        # ── Bottleneck ────────────────────────────────────────
        self.mid_res1 = ResidualBlock(ch4, ch4, time_emb_dim)
        self.mid_res2 = ResidualBlock(ch4, ch4, time_emb_dim)

        # ── Decoder ───────────────────────────────────────────
        # Level 3 up : ch4 + ch4 (skip) → ch4  (H/4 × W/4)
        self.up3       = Upsample(ch4)
        self.dec3_res1 = ResidualBlock(ch4 + ch4, ch4, time_emb_dim)
        self.dec3_res2 = ResidualBlock(ch4,        ch4, time_emb_dim)

        # Level 2 up : ch4 + ch2 (skip) → ch2  (H/2 × W/2)
        self.up2       = Upsample(ch4)
        self.dec2_res1 = ResidualBlock(ch4 + ch2, ch2, time_emb_dim)
        self.dec2_res2 = ResidualBlock(ch2,        ch2, time_emb_dim)

        # Level 1 up : ch2 + ch (skip)  → ch   (H × W)
        self.up1       = Upsample(ch2)
        self.dec1_res1 = ResidualBlock(ch2 + ch,  ch,  time_emb_dim)
        self.dec1_res2 = ResidualBlock(ch,         ch,  time_emb_dim)

        # ── Output Projection ─────────────────────────────────
        self.out_norm = nn.GroupNorm(8, ch)
        self.out_conv = nn.Conv2d(ch, in_channels, 1)   # predict 3-channel noise

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : (B, 3, H, W)   noisy image at timestep t
            t : (B,)           integer timesteps
        Returns:
            predicted noise (B, 3, H, W)
        """
        # Time embedding shared across all levels
        t_emb = self.time_emb(t)              # (B, time_emb_dim)

        # ── Encoder ───────────────────────────────────────────
        x0   = self.init_conv(x)

        e1   = self.enc1_res2(self.enc1_res1(x0,  t_emb), t_emb)   # ch
        e1d  = self.down1(e1)

        e2   = self.enc2_res2(self.enc2_res1(e1d, t_emb), t_emb)   # ch2
        e2d  = self.down2(e2)

        e3   = self.enc3_res2(self.enc3_res1(e2d, t_emb), t_emb)   # ch4
        e3d  = self.down3(e3)

        # ── Bottleneck ────────────────────────────────────────
        b    = self.mid_res2(self.mid_res1(e3d, t_emb), t_emb)     # ch4

        # ── Decoder (concat skip connections) ─────────────────
        d3   = self.up3(b)
        d3   = self.dec3_res2(self.dec3_res1(torch.cat([d3, e3], dim=1), t_emb), t_emb)

        d2   = self.up2(d3)
        d2   = self.dec2_res2(self.dec2_res1(torch.cat([d2, e2], dim=1), t_emb), t_emb)

        d1   = self.up1(d2)
        d1   = self.dec1_res2(self.dec1_res1(torch.cat([d1, e1], dim=1), t_emb), t_emb)

        # ── Output ────────────────────────────────────────────
        out  = self.out_conv(F.silu(self.out_norm(d1)))
        return out


# Instantiate & optionally wrap in DataParallel for dual T4
model = UNet(in_channels=3, base_channels=cfg.BASE_CHANNELS).to(cfg.DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel.")

total_params = sum(p.numel() for p in model.parameters())
print(f"U-Net parameters: {total_params:,}")

---
## PART 3 — Training Loop

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 3.1 — Optimiser, Scaler & (Optional) LR Scheduler
# ─────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=1e-4)

# Cosine LR scheduler (optional but recommended)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.EPOCHS, eta_min=1e-6
)

# Mixed precision scaler
scaler = GradScaler()

# MSE loss — compare predicted noise with actual added noise
criterion = nn.MSELoss()

print("Optimiser   : AdamW")
print("Scheduler   : CosineAnnealingLR")
print("Loss        : MSE (predicted ε  vs  actual ε)")
print("Mixed prec  : enabled via torch.amp")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 3.2 — Training Loop
# ─────────────────────────────────────────────────────────────
import gc  # Added for memory management

epoch_losses = []   # for the loss vs epochs plot

for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for step, x0 in enumerate(loader):
        x0 = x0.to(cfg.DEVICE)                              # clean images

        # Sample random timesteps, one per image in the batch
        t = noise_schedule.sample_timesteps(x0.size(0))     # (B,)

        # Forward diffusion: corrupt x0 → x_t  and keep true noise ε
        x_t, eps_true = noise_schedule.q_sample(x0, t)      # (B,C,H,W)

        optimizer.zero_grad(set_to_none=True)

        # ── Mixed precision forward pass ──────────────────────
        with torch.amp.autocast('cuda'):
            eps_pred = model(x_t, t)                         # U-Net predicts noise
            loss     = criterion(eps_pred, eps_true)         # MSE(ε̂, ε)

        # ── Backward + optimiser step (with GradScaler) ───────
        scaler.scale(loss).backward()

        # Optional gradient clipping to stabilise training
        if cfg.GRAD_CLIP is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    # ── End-of-epoch bookkeeping ───────────────────────────────
    avg_loss = running_loss / len(loader)
    epoch_losses.append(avg_loss)
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:3d}/{cfg.EPOCHS}]  loss = {avg_loss:.5f}  "
              f"lr = {scheduler.get_last_lr()[0]:.2e}")
              
    # ── Force memory cleanup at the end of every epoch ─────────
    gc.collect()
    torch.cuda.empty_cache()

# Save model weights
torch.save(model.state_dict(), cfg.SAVE_PATH)
print(f"\nModel saved to {cfg.SAVE_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 3.3 — Loss vs Epochs Plot
# ─────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses,
         color="steelblue", linewidth=2, marker="o", markersize=3)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("MSE Loss", fontsize=12)
plt.title("DDPM Training Loss", fontsize=14)
plt.grid(True, alpha=0.35)
plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()

---
## PART 4 — Sampling, Generation & Metrics

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 4.1 — DDPM Reverse Sampler
# ─────────────────────────────────────────────────────────────
@torch.no_grad()
def ddpm_sample(model: nn.Module, schedule: LinearNoiseSchedule,
                n_images: int = 1, img_size: int = 128,
                device: str = "cuda",
                capture_steps: list = None):
    """
    Reverse diffusion: start from x_T ~ N(0,I) and iteratively denoise
    down to x_0 using the DDPM update rule:

        x_{t-1} = (1/√α_t) · (x_t - (β_t/√(1-ᾱ_t)) · ε_θ(x_t, t))
                  + σ_t · z,   z ~ N(0,I)  (only when t > 1)

    Args:
        model        : trained UNet
        schedule     : LinearNoiseSchedule
        n_images     : number of images to generate simultaneously
        img_size     : spatial resolution
        device       : 'cuda' or 'cpu'
        capture_steps: list of timesteps at which to save intermediate images
                       (used for visualisation)
    Returns:
        x0_final          : (n_images, 3, H, W) final generated images
        intermediates     : dict[t] → tensor, only if capture_steps given
    """
    model.eval()
    capture_steps = set(capture_steps or [])
    intermediates = {}

    # Start from pure Gaussian noise
    x = torch.randn(n_images, 3, img_size, img_size).to(device)

    for t_val in reversed(range(1, schedule.T)):
        t_tensor = torch.full((n_images,), t_val, device=device, dtype=torch.long)

        # Predict noise with U-Net
        with torch.amp.autocast('cuda'):
            eps_pred = model(x, t_tensor)

        # Retrieve schedule quantities for this timestep
        beta_t         = schedule.betas[t_val]
        alpha_t        = schedule.alphas[t_val]
        alpha_hat_t    = schedule.alpha_hat[t_val]
        sqrt_1m_aht    = schedule.sqrt_one_minus_alpha_hat[t_val]

        # DDPM mean estimate
        coeff = beta_t / sqrt_1m_aht
        mean  = (1.0 / torch.sqrt(alpha_t)) * (x - coeff * eps_pred)

        # Add posterior noise (skip at t=1 to avoid extra noise on final step)
        if t_val > 1:
            noise   = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t)
            x       = mean + sigma_t * noise
        else:
            x = mean

        if t_val in capture_steps:
            intermediates[t_val] = x.clone().cpu()

    return x.cpu(), intermediates

print("DDPM sampler defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 4.2 — Generate 5 Images & Visualise Denoising Steps
# ─────────────────────────────────────────────────────────────
# Evenly-spaced timesteps to capture during reverse diffusion
vis_steps = np.linspace(cfg.T - 1, 1, 7, dtype=int).tolist()

print("Generating images… (this may take a few minutes)")
generated, intermediates = ddpm_sample(
    model, noise_schedule,
    n_images    = 5,
    img_size    = cfg.IMG_SIZE,
    device      = cfg.DEVICE,
    capture_steps = vis_steps,
)
print("Done.")

# ── Plot 5 generated images ────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
fig.suptitle("Generated Images (from pure noise)", fontsize=13)
for i in range(5):
    axes[i].imshow(tensor_to_numpy_img(generated[i]))
    axes[i].set_title(f"Sample {i+1}", fontsize=9)
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("generated_images.png", bbox_inches="tight")
plt.show()

# ── Plot reverse denoising steps for sample 0 ─────────────────
step_keys = sorted(intermediates.keys(), reverse=True)
fig2, axes2 = plt.subplots(1, len(step_keys), figsize=(3 * len(step_keys), 3.5))
fig2.suptitle("Reverse Diffusion  (noise → image)", fontsize=13, y=1.02)
for ax, t_key in zip(axes2, step_keys):
    ax.imshow(tensor_to_numpy_img(intermediates[t_key][0]))
    ax.set_title(f"t = {t_key}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig("reverse_diffusion_steps.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 4.3 — Image Reconstruction (Core Task)
#  Strategy: add a controlled amount of noise to the target
#  image at timestep t*, then denoise back from t* → 0.
# ─────────────────────────────────────────────────────────────
@torch.no_grad()
def reconstruct_image(target_tensor: torch.Tensor,
                      model: nn.Module,
                      schedule: LinearNoiseSchedule,
                      t_start: int = 200,
                      device: str = "cuda"):
    """
    Partial reconstruction:
      1. Encode target image to timestep t_start using q_sample
      2. Denoise from t_start → 0 using the reverse process

    Args:
        target_tensor : (3, H, W) normalised tensor
        t_start       : how many steps of noise to add (lower = closer to target)
    Returns:
        reconstructed : (3, H, W) reconstructed image tensor
    """
    model.eval()
    x0  = target_tensor.unsqueeze(0).to(device)
    t_t = torch.tensor([t_start], device=device)

    # Add noise up to t_start
    x_noisy, _ = schedule.q_sample(x0, t_t)           # (1,3,H,W)
    x = x_noisy

    # Reverse from t_start → 1
    for t_val in reversed(range(1, t_start + 1)):
        t_tensor = torch.full((1,), t_val, device=device, dtype=torch.long)

        with torch.amp.autocast('cuda'):
            eps_pred = model(x, t_tensor)

        beta_t         = schedule.betas[t_val]
        alpha_t        = schedule.alphas[t_val]
        sqrt_1m_aht    = schedule.sqrt_one_minus_alpha_hat[t_val]
        coeff          = beta_t / sqrt_1m_aht
        mean           = (1.0 / torch.sqrt(alpha_t)) * (x - coeff * eps_pred)

        if t_val > 1:
            x = mean + torch.sqrt(beta_t) * torch.randn_like(x)
        else:
            x = mean

    return x.squeeze(0).cpu()


# ── Run reconstruction on a random dataset image ────────────
target_img   = dataset[0]          # pick any image as target
reconstructed = reconstruct_image(
    target_img, model, noise_schedule, t_start=200, device=cfg.DEVICE
)

# ── Side-by-side comparison ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(tensor_to_numpy_img(target_img))
axes[0].set_title("Target Image", fontsize=11)
axes[0].axis("off")
axes[1].imshow(tensor_to_numpy_img(reconstructed))
axes[1].set_title("Reconstructed", fontsize=11)
axes[1].axis("off")
plt.suptitle("Target vs Reconstructed", fontsize=13)
plt.tight_layout()
plt.savefig("reconstruction_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  Cell 4.4 — PSNR & SSIM Evaluation
# ─────────────────────────────────────────────────────────────
def compute_metrics(img1_tensor: torch.Tensor, img2_tensor: torch.Tensor):
    """
    Compute PSNR and SSIM between two [-1,1] image tensors.

    Args:
        img1_tensor, img2_tensor : (3, H, W) float tensors in [-1, 1]
    Returns:
        psnr (float), ssim (float)
    """
    # Convert to [0, 1] numpy arrays for scikit-image
    def to_np(t):
        t = ((t.detach().cpu().clamp(-1, 1) + 1) / 2).numpy()  # (C,H,W) in [0,1]
        return t.transpose(1, 2, 0)                              # (H,W,C)

    np1 = to_np(img1_tensor)
    np2 = to_np(img2_tensor)

    psnr_val = sk_psnr(np1, np2, data_range=1.0)
    ssim_val = sk_ssim(np1, np2, data_range=1.0, channel_axis=-1)
    return psnr_val, ssim_val


# Evaluate reconstruction quality
psnr, ssim = compute_metrics(target_img, reconstructed)
print(f"Reconstruction Metrics:")
print(f"  PSNR : {psnr:.2f} dB")
print(f"  SSIM : {ssim:.4f}")

# Also evaluate a purely generated image against a random real image
psnr_gen, ssim_gen = compute_metrics(dataset[1], generated[0])
print(f"\nGenerated vs Real Metrics (indicative only):")
print(f"  PSNR : {psnr_gen:.2f} dB")
print(f"  SSIM : {ssim_gen:.4f}")

---
## Summary of Outputs

| File | Description |
|---|---|
| `forward_diffusion.png` | 7-step forward noising visualisation |
| `training_loss.png` | Loss vs Epochs plot |
| `generated_images.png` | 5 images generated from pure noise |
| `reverse_diffusion_steps.png` | Intermediate denoising steps |
| `reconstruction_comparison.png` | Target vs Reconstructed side-by-side |
| `ddpm_model.pth` | Trained model weights |
| `app.py` | Gradio app for HF Spaces deployment |
| `requirements.txt` | Python dependencies for HF Spaces |